# NumCompute Quickstart Demo

This notebook demonstrates the main features of the NumCompute toolkit using only **plain Python and NumPy**. It is designed to support the Assignment 2.1 demo requirement by showing CSV loading, preprocessing, sorting/searching, ranking, statistics, gradients/Jacobian, metrics, pipeline usage, and vectorised-vs-loop benchmarks.

## 1. Setup and imports

This cell makes the notebook work whether it is opened from the project root or from inside the `demo/` folder.

In [ ]:
from pathlib import Path
import sys
import time
import platform

import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "numcompute").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from numcompute.io import IO
from numcompute.preprocessing import SimpleImputer, StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.sort_search import topk, quickselect, binary_search
from numcompute.ranks import rank, percentile, percentiles
from numcompute.stats import Stats, StreamingStats
from numcompute.metrics import Classification, Regression
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline
from numcompute.utils import sigmoid, softmax, logsumexp, euclidean_distance, cosine_similarity

np.set_printoptions(precision=3, suppress=True)
print("Project root:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)

## 2. Read CSV data using `io.py`

The CSV file contains numeric values, categorical values, booleans, and missing values. This demonstrates dtype inference and missing-value handling.

In [ ]:
csv_path = PROJECT_ROOT / "demo" / "numcompute_demo_data.csv"
csv_data = IO.load_csv(str(csv_path))

print(csv_data)
print("Data array:")
print(csv_data.data)
print("Column metadata:")
for col in csv_data.cols:
    print(f"{col.name}: {col.dtype}")

## 3. Split numeric and categorical features

The preprocessing module uses numeric transformations for numeric columns and one-hot encoding for categorical columns.

In [ ]:
numeric_idx = [i for i, col in enumerate(csv_data.cols) if col.dtype in ("int", "float")]
categorical_idx = [i for i, col in enumerate(csv_data.cols) if col.dtype == "str"]

X_numeric = csv_data.data[:, numeric_idx].astype(float)
X_category = csv_data.data[:, categorical_idx].astype(object)

print("Numeric feature matrix:")
print(X_numeric)
print("Categorical feature matrix:")
print(X_category)

## 4. Preprocessing with imputation, scaling, and one-hot encoding

This section demonstrates the preprocessing components: missing-value imputation, standard scaling, min-max scaling, and categorical one-hot encoding.

In [ ]:
imputer = SimpleImputer(strategy="mean")
X_filled = imputer.fit_transform(X_numeric)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)

minmax = MinMaxScaler()
X_minmax = minmax.fit_transform(X_filled)

encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X_category)

X_final = np.hstack([X_scaled, X_encoded])

print("After mean imputation:")
print(X_filled)
print("After standard scaling:")
print(X_scaled)
print("After min-max scaling:")
print(X_minmax)
print("One-hot encoded category columns:")
print(X_encoded)
print("Final combined feature matrix shape:", X_final.shape)
print(X_final)

## 5. Pipeline abstraction

The lightweight `Pipeline` class allows reusable transformations to be chained together with a consistent `fit_transform` workflow.

In [ ]:
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

X_pipeline = pipe.fit_transform(X_numeric)
print("Pipeline output:")
print(X_pipeline)

## 6. Sorting, top-k, quickselect, and binary search

This section demonstrates algorithmic components from `sort_search.py`.

In [ ]:
scores = csv_data.data[:, 2].astype(float)
scores_clean = scores[~np.isnan(scores)]

print("Scores without NaN:", scores_clean)

top_values, top_indices = topk(scores_clean, k=2, largest=True, return_indices=True)
print("Top 2 values:", top_values)
print("Top 2 indices:", top_indices)

sorted_scores = np.sort(scores_clean)
print("Sorted scores:", sorted_scores)

index, found = binary_search(sorted_scores, 88.0)
print("Binary search for 88.0 -> index:", index, "found:", found)

third_smallest = quickselect(scores_clean, k=2)
print("3rd smallest score using quickselect:", third_smallest)

## 7. Ranking and percentiles

Ranking demonstrates tie handling. Percentiles provide useful descriptive summaries of numerical arrays.

In [ ]:
print("Average ranks:", rank(scores_clean, method="average"))
print("Dense ranks:", rank(scores_clean, method="dense"))
print("Ordinal ranks:", rank(scores_clean, method="ordinal"))

print("50th percentile:", percentile(scores_clean, 50))
print("25th, 50th, 75th percentiles:", percentiles(scores_clean, [25, 50, 75]))

## 8. Descriptive and streaming statistics

`Stats` provides column-wise summaries, histograms, quantiles, and streaming mean/variance using Welford's algorithm.

In [ ]:
print("Column-wise means:")
print(Stats.mean(csv_data, axis=0))

print("Column-wise standard deviations:")
print(Stats.std(csv_data, axis=0))

print("Column-wise median/0.5 quantile:")
print(Stats.quantile(csv_data, 0.5, axis=0))

counts, edges = Stats.histogram(csv_data, bins=4)
print("Histogram counts:", counts)
print("Histogram bin edges:", edges)

print("Streaming mean over all numeric values:", Stats.streaming_mean(csv_data))
print("Streaming variance over all numeric values:", Stats.streaming_variance(csv_data, ddof=0))

stream = StreamingStats()
stream.update_many([1, 2, 3, np.nan, 4, 5])
print("Manual StreamingStats mean:", stream.mean)
print("Manual StreamingStats variance:", stream.variance(ddof=0))

## 9. Classification and regression metrics

This section demonstrates binary classification metrics and mean squared error.

In [ ]:
y_true = np.array([1, 0, 1, 1, 0, 1])
y_pred = np.array([1, 0, 0, 1, 0, 1])

print("Confusion matrix (tp, tn, fp, fn):", Classification.confusion_matrix(y_true, y_pred))
print("Accuracy:", Classification.accuracy(y_true, y_pred))
print("Precision:", Classification.precision(y_true, y_pred))
print("Recall:", Classification.recall(y_true, y_pred))
print("F1:", Classification.f1(y_true, y_pred))

actual = np.array([82.0, 76.0, 91.0, 88.0])
predicted = np.array([80.0, 78.0, 90.0, 89.0])
print("MSE:", Regression.mse(actual, predicted))

## 10. Finite-difference gradient and Jacobian

`optim.py` estimates gradients for scalar functions and Jacobians for vector-valued functions using finite differences.

In [ ]:
def objective(x):
    return x[0] ** 2 + 3 * x[1] ** 2 + 4 * x[2]

x0 = np.array([2.0, 3.0, 4.0])
print("Gradient at", x0, ":", grad(objective, x0, method="central"))


def vector_function(x):
    return np.array([
        x[0] + x[1] + x[2],
        x[0] ** 2,
        np.sin(x[1]),
    ])

print("Jacobian:")
print(jacobian(vector_function, x0, method="central"))

## 11. Numerical utilities

The utility module includes stable activation functions, logsumexp, distances, similarity, and batching helpers.

In [ ]:
large_values = np.array([-1000.0, 0.0, 1000.0])
print("Stable sigmoid:", sigmoid(large_values))

logits = np.array([1000.0, 1001.0, 1002.0])
print("Stable softmax:", softmax(logits))
print("Stable logsumexp:", logsumexp(logits))
print("All -inf logsumexp:", logsumexp(np.array([-np.inf, -np.inf])))

print("Euclidean distance:", euclidean_distance([1, 2, 3], [2, 2, 4]))
print("Cosine similarity:", cosine_similarity([1, 0, 1], [0, 1, 1]))

## 12. Benchmark: vectorised NumPy vs Python loops

This cell performs a small benchmark directly inside the notebook. The standalone version is available at `benchmark/run_benchmarks.py`.

In [ ]:
def loop_mean(values):
    total = 0.0
    count = 0
    for value in values:
        total += float(value)
        count += 1
    return total / count


def loop_mse(y_true, y_pred):
    total = 0.0
    count = 0
    for a, b in zip(y_true, y_pred):
        err = a - b
        total += err * err
        count += 1
    return total / count


def time_call(func, *args, repeats=5):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        func(*args)
        times.append(time.perf_counter() - start)
    return min(times)

rng = np.random.default_rng(42)
x = rng.normal(size=100_000)
y_true_b = rng.normal(size=100_000)
y_pred_b = y_true_b + rng.normal(scale=0.1, size=100_000)

benchmarks = [
    ("Mean", lambda arr: np.mean(arr), loop_mean, (x,)),
    ("MSE", lambda a, b: np.mean((a - b) ** 2), loop_mse, (y_true_b, y_pred_b)),
]

print(f"{'Task':<10} {'Vectorised (s)':<16} {'Loop (s)':<12} {'Speedup':<10}")
print("-" * 52)
for name, vec_func, loop_func, args in benchmarks:
    vec_time = time_call(vec_func, *args)
    loop_time = time_call(loop_func, *args)
    speedup = loop_time / vec_time if vec_time > 0 else np.inf
    print(f"{name:<10} {vec_time:<16.6f} {loop_time:<12.6f} {speedup:<10.2f}x")

## 13. Running tests

The command below is commented out so the notebook does not unexpectedly run a full test suite. Uncomment it when running from the project root if `pytest` is installed.

In [ ]:
!python -m pytest -q "$PROJECT_ROOT"/"tests"/*

## 14. Demo summary

This quickstart demonstrates the main assignment requirements:

- CSV loading with missing values
- preprocessing with imputation, scaling, and one-hot encoding
- pipeline abstraction
- top-k, quickselect, and binary search
- ranking and percentiles
- descriptive statistics, histograms, quantiles, and Welford streaming stats
- classification and regression metrics
- finite-difference gradient and Jacobian
- stable numerical utilities
- benchmark comparison between vectorised NumPy and Python loops